In [ ]:
Quiz2.py
========
UCI Credit Card Default Dataset - Multi-Layer Perceptron (PyTorch)

Task requirements covered:
  1. Feature Scaling      -> StandardScaler on all numeric input columns
  2. MLP model             -> built with PyTorch (torch.nn)
  3. Single validation sample prediction -> shown explicitly at the bottom
  4. Training >= 50 epochs -> trained for EPOCHS (default 50) with Training/Validation Loss plot
  5. Hyperparameter explanation -> printed to console + written as comments below

Expected data path (per your project layout):
    /Data/UCI_Credit_Card.csv

Run:
    python Quiz2.py
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

# --------------------------------------------------------------------------
# 0. Reproducibility
# --------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --------------------------------------------------------------------------
# 1. Hyperparameters  (all in one place so they're easy to explain/tune)
# --------------------------------------------------------------------------
DATA_PATH     = "/Data/UCI_Credit_Card.csv"   # dataset location
TARGET_COL    = "default.payment.next.month"  # label column in this dataset
TEST_SIZE     = 0.2                           # 20% held out as validation set

EPOCHS        = 50          # number of full passes over the training data
BATCH_SIZE    = 64          # number of samples per gradient update
LEARNING_RATE = 1e-3        # step size for the optimizer
HIDDEN_LAYERS = [64, 32, 16]  # number of neurons in each hidden layer
DROPOUT_RATE  = 0.2          # dropout probability (regularization)
OPTIMIZER_NAME = "Adam"      # optimization algorithm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------------------------------
# 2. Load data
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Drop the ID column if present (not a predictive feature)
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

X = df.drop(columns=[TARGET_COL]).values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)

feature_names = df.drop(columns=[TARGET_COL]).columns.tolist()
print(f"Loaded data: {X.shape[0]} samples, {X.shape[1]} features.")

# --------------------------------------------------------------------------
# 3. Train / Validation split
# --------------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# --------------------------------------------------------------------------
# 4. Feature Scaling
#    Fit the scaler ONLY on the training data, then apply the same
#    transform to the validation data (prevents data leakage).
#    This is important because columns like LIMIT_BAL / BILL_AMT are
#    in the tens-of-thousands range while PAY_0..PAY_6 are small
#    integers (-2..8) -> without scaling, large-magnitude columns
#    would dominate the gradient updates.
# --------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# --------------------------------------------------------------------------
# 5. PyTorch Dataset / DataLoader
# --------------------------------------------------------------------------
class CreditCardDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = CreditCardDataset(X_train_scaled, y_train)
val_dataset   = CreditCardDataset(X_val_scaled, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --------------------------------------------------------------------------
# 6. Model: Multi-Layer Perceptron
# --------------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # output logit (binary classification)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # raw logits; BCEWithLogitsLoss applies sigmoid internally


model = MLP(input_dim=X_train_scaled.shape[1],
            hidden_layers=HIDDEN_LAYERS,
            dropout_rate=DROPOUT_RATE).to(DEVICE)

print(model)

# --------------------------------------------------------------------------
# 7. Loss & Optimizer
# --------------------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --------------------------------------------------------------------------
# 8. Training loop
# --------------------------------------------------------------------------
train_losses = []
val_losses = []

for epoch in range(1, EPOCHS + 1):
    # ---- training ----
    model.train()
    running_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * xb.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # ---- validation ----
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_val_loss += loss.item() * xb.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)

    print(f"Epoch [{epoch:3d}/{EPOCHS}]  "
          f"Train Loss: {epoch_train_loss:.4f}  |  Val Loss: {epoch_val_loss:.4f}")

# --------------------------------------------------------------------------
# 9. Validation-set metrics
# --------------------------------------------------------------------------
model.eval()
with torch.no_grad():
    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(DEVICE)
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits).cpu().numpy().flatten()
    val_preds = (val_probs >= 0.5).astype(int)

acc = accuracy_score(y_val, val_preds)
auc = roc_auc_score(y_val, val_probs)
print(f"\nValidation Accuracy: {acc:.4f}")
print(f"Validation AUC:      {auc:.4f}")

# --------------------------------------------------------------------------
# 10. Plot Training Loss vs Validation Loss
# --------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Training Loss")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (BCE)")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

out_dir = "/Codes"
os.makedirs(out_dir, exist_ok=True) if os.access("/", os.W_OK) else None
plot_path = os.path.join(out_dir, "loss_curve.png") if os.path.isdir(out_dir) else "loss_curve.png"
plt.savefig(plot_path, dpi=150)
print(f"\nLoss curve saved to: {plot_path}")
plt.show()

# --------------------------------------------------------------------------
# 11. Predict a single Validation sample (explicit demonstration)
# --------------------------------------------------------------------------
sample_idx = 0  # change this index to inspect a different validation sample

sample_x = X_val_scaled[sample_idx]          # already scaled
sample_x_raw = X_val[sample_idx]             # original (unscaled) values, for display
sample_y_true = y_val[sample_idx]

sample_tensor = torch.tensor(sample_x, dtype=torch.float32).unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    logit = model(sample_tensor)
    prob = torch.sigmoid(logit).item()
    pred_label = 1 if prob >= 0.5 else 0

print("\n" + "=" * 60)
print("Single Validation Sample Prediction")
print("=" * 60)
print("Raw feature values (first 10 shown):")
for name, val in list(zip(feature_names, sample_x_raw))[:10]:
    print(f"  {name:20s}: {val:.2f}")
print(f"\nTrue label:           {int(sample_y_true)}  "
      f"({'Default' if sample_y_true == 1 else 'No Default'})")
print(f"Predicted probability: {prob:.4f}")
print(f"Predicted label:       {pred_label}  "
      f"({'Default' if pred_label == 1 else 'No Default'})")
print("=" * 60)

# --------------------------------------------------------------------------
# 12. Hyperparameter Summary / Explanation
# --------------------------------------------------------------------------
print("""
Hyperparameter Explanation
---------------------------
Epochs (50):
    Number of times the entire training set is passed through the network.
    More epochs let the model learn more, but too many can lead to
    overfitting (visible as validation loss rising while training loss
    keeps falling) -- watch the Training/Validation Loss plot for this.

Batch Size (64):
    Number of samples processed before the model's weights are updated once.
    Smaller batches -> noisier but more frequent updates (can help
    generalization); larger batches -> smoother, faster-per-epoch updates
    but need more memory.

Learning Rate (1e-3):
    Step size used by the optimizer when updating weights. Too high ->
    training is unstable / loss diverges. Too low -> training converges
    very slowly. 1e-3 is a common, robust default for Adam.

Optimizer (Adam):
    Adaptive Moment Estimation -- combines momentum with per-parameter
    adaptive learning rates. Generally converges faster and needs less
    manual tuning than plain SGD, which is why it's used here.

Hidden Layers / Neurons ([64, 32, 16]):
    Three hidden layers with decreasing width (64 -> 32 -> 16), each
    followed by ReLU activation and Dropout(0.2). The funnel shape lets
    the network first learn broad feature combinations, then compress
    them into higher-level, more abstract representations before the
    final output layer produces a single logit for binary classification.

Dropout (0.2):
    Randomly zeroes 20% of neuron activations during training to reduce
    overfitting by preventing co-adaptation of neurons.
""")